# Chroma Integration Test

Thorough end-to-end test of the ChromaDB corpus integration. Exercises every layer:

1. **Setup** — Imports, path config, sample docs
2. **ChromaChunkSource** — Populate, sample, context, search (vector / lexical / hybrid)
3. **Filter mapper** — Predicate AST → Chroma filters
4. **Pickle safety** — Roundtrip serialize/deserialize
5. **SearchEnv** — RL tool execution
6. **CgftPipeline** — Full synthetic QA generation with Chroma backend

Uses an **in-memory** Chroma client so no server is needed.

## 1. Setup

In [1]:
import sys
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")

# Should show the worktree venv path:
# .claude/worktrees/chroma/.venv/bin/python
# If it shows something else, the wrong kernel is selected.

import cgft
print(f"cgft: {cgft.__file__}")

Python: /Users/thariqridha/.venvs/cgft-chroma/bin/python
Version: 3.12.12 (main, Feb 20 2026, 11:13:39) [Clang 17.0.0 (clang-1700.0.13.5)]
cgft: /Users/thariqridha/Projects/cgft_projects/cgft/.claude/worktrees/chroma/src/cgft/__init__.py


In [2]:
import cgft
from cgft.utils import ensure_safe_python_version

ensure_safe_python_version()
print(f"cgft loaded from: {cgft.__file__}")

cgft loaded from: /Users/thariqridha/Projects/cgft_projects/cgft/.claude/worktrees/chroma/src/cgft/__init__.py


In [3]:
# Verify chromadb is installed
import chromadb
print(f"chromadb version: {chromadb.__version__}")

# Check if the new Search API is available
try:
    from chromadb import Knn, Search
    SEARCH_API_AVAILABLE = True
    print("Search API: AVAILABLE (lexical + hybrid modes enabled)")
except ImportError:
    SEARCH_API_AVAILABLE = False
    print("Search API: NOT AVAILABLE (vector-only mode)")

chromadb version: 1.5.5
Search API: AVAILABLE (lexical + hybrid modes enabled)


In [4]:
# Create sample markdown docs for testing (no external files needed)
import tempfile, os

DOCS_DIR = tempfile.mkdtemp(prefix="chroma_test_docs_")

docs = {
    "getting_started.md": """\
# Getting Started

## Installation

Install the package using pip:

```bash
pip install cgft
```

Make sure you have Python 3.12 installed. Python 3.13 is not supported due to
a pathlib.Path pickle incompatibility.

## Quick Start

Create a new project directory and initialize your first experiment:

```python
from cgft.corpus.chroma import ChromaChunkSource

source = ChromaChunkSource(collection_name="my-docs", host="localhost")
source.populate_from_folder("./docs")
```

The system will automatically chunk your documents, compute embeddings, and
upload them to ChromaDB for retrieval during training.
""",
    "features/search.md": """\
# Search Features

## Full-Text Search

ChromaDB supports native BM25 full-text search via sparse vector indexes.
This allows keyword-based retrieval without requiring dense embeddings.

### Configuration

Enable BM25 by setting `enable_bm25=True` (the default) when creating
your ChromaChunkSource. The system will automatically create a sparse
vector index using the BM25 embedding function.

## Vector Search

Dense vector similarity search uses cosine distance by default. You can
provide a custom embedding function or use Chroma's built-in
sentence-transformers model.

### Supported Distance Metrics

- **cosine** (default) - Angular similarity, good for normalized embeddings
- **l2** - Euclidean distance, better for unnormalized vectors
- **ip** - Inner product, fastest but requires normalized vectors

## Hybrid Search

Combine BM25 and vector search using Reciprocal Rank Fusion (RRF).
This gives you the best of both keyword matching and semantic similarity.
""",
    "features/environments.md": """\
# RL Environments

## Overview

Environments define how the RL agent interacts with your corpus during
training. Each environment provides tools the agent can use and a reward
function that scores the agent's responses.

## SearchEnv

The `SearchEnv` extends `SearchEnv` to provide a search tool backed
by ChromaDB. It supports vector, lexical (BM25), and hybrid search modes.

### Requirements

- Client-server mode only (host/port required)
- ChromaDB server must be accessible from the training cluster

### Reward Function

Uses an LLM judge to evaluate answer correctness against ground truth.
The judge scores responses on a 0-1 scale based on factual consistency.

## Custom Environments

You can create custom environments by extending `SearchEnv` and defining
your own tools and reward functions. See the benchmax documentation for
the `BaseEnv` interface.
""",
    "api/chunking.md": """\
# Chunking API

## MarkdownChunker

The `MarkdownChunker` splits documents using a 3-stage pipeline:

1. **Header splitting** - Split by H1/H2/H3 headers, preserving hierarchy
2. **Fusion** - Merge adjacent short sections to avoid over-fragmentation
3. **Recursive splitting** - Break large sections with configurable overlap

### Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| min_char  | 1024    | Minimum characters per chunk |
| max_char  | 2048    | Maximum characters per chunk |
| chunk_overlap | 128 | Character overlap between chunks |

### Example

```python
from cgft.chunkers.markdown import MarkdownChunker

chunker = MarkdownChunker(min_char=512, max_char=1024, chunk_overlap=64)
collection = chunker.chunk_folder("./docs")
print(f"Created {len(collection)} chunks from {len(collection.files)} files")
```
""",
}

for name, content in docs.items():
    filepath = os.path.join(DOCS_DIR, name)
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w") as f:
        f.write(content)

print(f"Created {len(docs)} sample docs in {DOCS_DIR}")
for name in sorted(docs):
    print(f"  {name} ({len(docs[name])} chars)")

Created 4 sample docs in /var/folders/8w/8rh15bd90x7bn744lzw9xrhm0000gn/T/chroma_test_docs_qsxldsku
  api/chunking.md (863 chars)
  features/environments.md (878 chars)
  features/search.md (973 chars)
  getting_started.md (608 chars)


## 2. ChromaChunkSource — Populate & Inspect

Create an in-memory ChromaChunkSource and populate it from our sample docs.

In [5]:
from cgft.corpus.chroma.source import ChromaChunkSource

# Use in-memory client for testing (no server needed)
source = ChromaChunkSource(
    collection_name="integration-test",
    # No host/path = ephemeral in-memory
    enable_bm25=True,  # will only take effect if Search API available
)

print(f"Client-server mode: {source.is_client_server}")
print(f"Search capabilities: {source.get_search_capabilities()}")

Client-server mode: False
Search capabilities: {'backend': 'chroma', 'modes': {'vector'}, 'filter_ops': {'field': {'eq', 'in', 'gte', 'lte'}, 'logical': {'not', 'and', 'or'}}, 'ranking': {'cosine'}, 'constraints': {'max_top_k': 10000, 'vector_dimensions': None}, 'graph_expansion': False}


In [6]:
# Populate from the sample docs folder
# Uses small chunk sizes to generate more chunks from our short test docs
source.populate_from_folder(
    DOCS_DIR,
    min_chars=200,
    max_chars=600,
    overlap_chars=50,
    file_extensions=[".md"],
)

Chunking documents from /var/folders/8w/8rh15bd90x7bn744lzw9xrhm0000gn/T/chroma_test_docs_qsxldsku...
ChunkCollection Summary
Total chunks: 11
Total files: 4

Chunk Size Statistics:
  Min: 159 chars (features/search.md, chunk 2)
  Max: 419 chars (features/search.md, chunk 1)
  Mean: 302 chars

File Structure:
├── features/
│   ├── environments.md (3 chunks)
│   └── search.md (3 chunks)
├── api/
│   └── chunking.md (3 chunks)
└── getting_started.md (2 chunks)

Uploading 11 chunks to Chroma collection 'integration-test'...


Uploading batches:   0%|          | 0/1 [00:00<?, ?it/s]


Upload complete! 11 chunks written to collection 'integration-test'.


In [7]:
# Verify total count
total = source._get_total_count()
print(f"Total chunks in collection: {total}")
assert total > 0, "No chunks were uploaded!"

Total chunks in collection: 11


### 2a. sample_chunks()

In [8]:
# Sample without min_chars filter
sampled = source.sample_chunks(n=3)
print(f"Sampled {len(sampled)} chunks (no min_chars filter):")
for i, chunk in enumerate(sampled):
    print(f"\n--- Chunk {i+1} ({len(chunk.content)} chars) ---")
    print(chunk.content[:150] + "...")
    print(f"Metadata: {chunk.metadata_dict}")

assert len(sampled) == 3

Sampled 3 chunks (no min_chars filter):

--- Chunk 1 (298 chars) ---
## SearchEnv  
The `SearchEnv` extends `SearchEnv` to provide a search tool backed
by ChromaDB. It supports vector, lexical (BM25), and hy...
Metadata: {'h3': 'Requirements', 'chunk_hash': '5af74478f32632b9d457e42d717ad9ea98a2ddf1f0aae04531b85323e061070c', 'file_path': 'features/environments.md', 'char_count': 298, 'h1': 'RL Environments', 'chunk_index': 1, 'h2': 'SearchEnv'}

--- Chunk 2 (396 chars) ---
# Search Features  
## Full-Text Search  
ChromaDB supports native BM25 full-text search via sparse vector indexes.
This allows keyword-based retrieva...
Metadata: {'h1': 'Search Features', 'h2': 'Full-Text Search', 'char_count': 396, 'h3': 'Configuration', 'chunk_hash': '0553cba42caecec0212a0fc78ed808b1f7e54a523c1cfa577cbbdb4ca2bad707', 'file_path': 'features/search.md', 'chunk_index': 0}

--- Chunk 3 (261 chars) ---
### Parameters  
| Parameter | Default | Description |
|-----------|---------|-------------|
| min_ch

In [9]:
# Sample with min_chars filter
sampled_long = source.sample_chunks(n=2, min_chars=300)
print(f"Sampled {len(sampled_long)} chunks (min_chars=300):")
for chunk in sampled_long:
    print(f"  - {len(chunk.content)} chars: {chunk.content[:80]}...")
    assert len(chunk.content) >= 300, f"Chunk too short: {len(chunk.content)} chars"
print("All chunks meet min_chars requirement.")

Sampled 2 chunks (min_chars=300):
  - 394 chars: ## Quick Start  
Create a new project directory and initialize your first experi...
  - 360 chars: ### Reward Function  
Uses an LLM judge to evaluate answer correctness against g...
All chunks meet min_chars requirement.


### 2b. File awareness — context & top-level chunks

In [10]:
# Test file awareness
print(f"File-aware: {source._files.check()}")

# Get all file paths
all_paths = source._files.get_all_file_paths()
print(f"\nAll file paths ({len(all_paths)}):")
for p in all_paths:
    print(f"  {p}")

assert len(all_paths) > 0, "No file paths found!"

File-aware: True

All file paths (4):
  api/chunking.md
  features/environments.md
  features/search.md
  getting_started.md


In [11]:
# Test get_top_level_chunks
top_level = source.get_top_level_chunks()
print(f"Top-level chunks: {len(top_level)}")
for chunk in top_level[:3]:
    fp = source._files.chunk_file_path(chunk)
    idx = source._files.chunk_index(chunk)
    print(f"  file={fp}, index={idx}, len={len(chunk.content)}")

Top-level chunks: 2
  file=getting_started.md, index=0, len=218
  file=getting_started.md, index=1, len=394


In [12]:
# Test get_chunk_with_context
test_chunk = sampled[0]
context = source.get_chunk_with_context(test_chunk, max_chars=100)
print("Chunk with context:")
print(f"  chunk_content length: {len(context['chunk_content'])}")
print(f"  prev_chunk_preview: {context['prev_chunk_preview'][:80]}...")
print(f"  next_chunk_preview: {context['next_chunk_preview'][:80]}...")

Chunk with context:
  chunk_content length: 547
  prev_chunk_preview: {
  "char_count": 221,
  "chunk_hash": "474e507183a5fd289519c1732b155752212bb541...
  next_chunk_preview: {
  "char_count": 360,
  "chunk_hash": "1b4bf5b2ac4c7c9c7e2bf9feb2b37212f7493f81...


### 2c. Vector search

In [13]:
from cgft.corpus.search_schema.search_types import SearchSpec

# Vector search using text query (Chroma auto-embeds)
spec = SearchSpec(mode="vector", top_k=3, text_query="how to install")
results = source.search(spec)

print(f"Vector search for 'how to install' — {len(results)} results:")
for i, chunk in enumerate(results):
    print(f"\n  Result {i+1} ({len(chunk.content)} chars):")
    print(f"    {chunk.content[:120]}...")

assert len(results) > 0, "Vector search returned no results!"
print("\nVector search: PASSED")

Vector search for 'how to install' — 3 results:

  Result 1 (218 chars):
    # Getting Started  
## Installation  
Install the package using pip:  
```bash
pip install cgft
```  
Make sure you have...

  Result 2 (298 chars):
    ## SearchEnv  
The `SearchEnv` extends `SearchEnv` to provide a search tool backed
by ChromaDB. It supports ...

  Result 3 (221 chars):
    # RL Environments  
## Overview  
Environments define how the RL agent interacts with your corpus during
training. Each ...

Vector search: PASSED


In [14]:
# search_content() — returns strings, not Chunks (pickle-safe for envs)
content_results = source.search_content(spec)
print(f"search_content() returned {len(content_results)} strings:")
for i, text in enumerate(content_results):
    print(f"  {i+1}. {text[:100]}...")
    assert isinstance(text, str), f"Expected str, got {type(text)}"

print("\nsearch_content: PASSED")

search_content() returned 3 strings:
  1. # Getting Started  
## Installation  
Install the package using pip:  
```bash
pip install cgft
``` ...
  2. ## SearchEnv  
The `SearchEnv` extends `SearchEnv` to provide a search tool backed
by Ch...
  3. # RL Environments  
## Overview  
Environments define how the RL agent interacts with your corpus du...

search_content: PASSED


In [15]:
# search_text() — convenience method, picks best available mode
text_results = source.search_text("distance metrics cosine", top_k=3)
print(f"search_text() returned {len(text_results)} chunks:")
for i, chunk in enumerate(text_results):
    print(f"  {i+1}. ({len(chunk.content)} chars) {chunk.content[:100]}...")

assert len(text_results) > 0
print("\nsearch_text: PASSED")

search_text() returned 3 chunks:
  1. (419 chars) ## Vector Search  
Dense vector similarity search uses cosine distance by default. You can
provide a...
  2. (298 chars) ## SearchEnv  
The `SearchEnv` extends `SearchEnv` to provide a search tool backed
by Ch...
  3. (159 chars) ## Hybrid Search  
Combine BM25 and vector search using Reciprocal Rank Fusion (RRF).
This gives you...

search_text: PASSED


In [16]:
# search_related() — multi-query deduplication with file-awareness
anchor = sampled[0]
print(f"Anchor chunk: {anchor.content[:80]}...")
print(f"  file: {source._files.chunk_file_path(anchor)}")

related = source.search_related(
    source=anchor,
    queries=["installation guide", "getting started tutorial"],
    top_k=3,
)

print(f"\nsearch_related() returned {len(related)} related chunks:")
for entry in related:
    c = entry["chunk"]
    print(f"  - queries={entry['queries']}, same_file={entry['same_file']}, "
          f"score={entry['max_score']:.3f}")
    print(f"    {c.content[:80]}...")

print("\nsearch_related: PASSED")

Anchor chunk: ## SearchEnv  
The `SearchEnv` extends `SearchEnv` to provide a sear...
  file: features/environments.md

search_related() returned 2 related chunks:
  - queries=['getting started tutorial'], same_file=False, score=0.538
    ## Quick Start  
Create a new project directory and initialize your first experi...
  - queries=['installation guide'], same_file=False, score=0.534
    # Getting Started  
## Installation  
Install the package using pip:  
```bash
p...

search_related: PASSED


### 2d. Search with metadata filters

In [17]:
from cgft.corpus.search_schema.search_types import FieldPredicate, AndPredicate

# Search with a metadata filter — only chunks with h1="Search Features"
filtered_spec = SearchSpec(
    mode="vector",
    top_k=5,
    text_query="BM25 keyword search",
    filter=FieldPredicate(field="h1", op="eq", value="Search Features"),
)
filtered_results = source.search(filtered_spec)

print(f"Filtered search (h1='Search Features') — {len(filtered_results)} results:")
for chunk in filtered_results:
    h1 = chunk.get_metadata("h1", "")
    print(f"  h1={h1!r} | {chunk.content[:80]}...")

# All results should have the correct h1
for chunk in filtered_results:
    assert chunk.get_metadata("h1") == "Search Features", \
        f"Filter leak: got h1={chunk.get_metadata('h1')!r}"

print("\nFiltered search: PASSED")

Filtered search (h1='Search Features') — 3 results:
  h1='Search Features' | ## Hybrid Search  
Combine BM25 and vector search using Reciprocal Rank Fusion (...
  h1='Search Features' | # Search Features  
## Full-Text Search  
ChromaDB supports native BM25 full-tex...
  h1='Search Features' | ## Vector Search  
Dense vector similarity search uses cosine distance by defaul...

Filtered search: PASSED


### 2e. Lexical & hybrid search (if Search API available)

In [18]:
caps = source.get_search_capabilities()

if "lexical" in caps["modes"]:
    # Test lexical (BM25) search
    lex_spec = SearchSpec(mode="lexical", top_k=3, text_query="pip install cgft")
    lex_results = source.search(lex_spec)
    print(f"Lexical search for 'pip install cgft' — {len(lex_results)} results:")
    for i, chunk in enumerate(lex_results):
        print(f"  {i+1}. {chunk.content[:100]}...")
    assert len(lex_results) > 0, "Lexical search returned no results!"
    print("Lexical search: PASSED\n")
else:
    print("Lexical search: SKIPPED (Search API not available)")

if "hybrid" in caps["modes"]:
    # Test hybrid (RRF) search
    hybrid_spec = SearchSpec(
        mode="hybrid",
        top_k=3,
        text_query="custom environments reward function",
    )
    hybrid_results = source.search(hybrid_spec)
    print(f"Hybrid search for 'custom environments reward function' — {len(hybrid_results)} results:")
    for i, chunk in enumerate(hybrid_results):
        print(f"  {i+1}. {chunk.content[:100]}...")
    assert len(hybrid_results) > 0, "Hybrid search returned no results!"
    print("Hybrid search: PASSED")
else:
    print("Hybrid search: SKIPPED (Search API not available)")

Lexical search: SKIPPED (Search API not available)
Hybrid search: SKIPPED (Search API not available)


## 3. Filter Mapper

Test the predicate AST → Chroma filter dict translation directly.

In [19]:
from cgft.corpus.chroma.filter_mapper import to_chroma_filters
from cgft.corpus.search_schema.search_types import (
    FieldPredicate, AndPredicate, OrPredicate, NotPredicate,
)

caps = source.get_search_capabilities()

# Simple field predicate
result = to_chroma_filters(FieldPredicate(field="h1", op="eq", value="Search"), caps)
assert result == {"h1": {"$eq": "Search"}}, f"Unexpected: {result}"
print(f"eq:  {result}")

# AND predicate
result = to_chroma_filters(
    AndPredicate(clauses=(
        FieldPredicate(field="h1", op="eq", value="X"),
        FieldPredicate(field="char_count", op="gte", value=100),
    )),
    caps,
)
assert "$and" in result
print(f"and: {result}")

# OR predicate
result = to_chroma_filters(
    OrPredicate(clauses=(
        FieldPredicate(field="h1", op="eq", value="A"),
        FieldPredicate(field="h1", op="eq", value="B"),
    )),
    caps,
)
assert "$or" in result
print(f"or:  {result}")

# NOT(eq) → $ne emulation
result = to_chroma_filters(
    NotPredicate(clause=FieldPredicate(field="h1", op="eq", value="skip")),
    caps,
)
assert result == {"h1": {"$ne": "skip"}}, f"Unexpected: {result}"
print(f"not: {result}")

# None → None
assert to_chroma_filters(None, caps) is None
print(f"none: None")

print("\nFilter mapper: ALL PASSED")

eq:  {'h1': {'$eq': 'Search'}}
and: {'$and': [{'h1': {'$eq': 'X'}}, {'char_count': {'$gte': 100}}]}
or:  {'$or': [{'h1': {'$eq': 'A'}}, {'h1': {'$eq': 'B'}}]}
not: {'h1': {'$ne': 'skip'}}
none: None

Filter mapper: ALL PASSED


## 4. Pickle Safety

Verify the source survives a pickle roundtrip (critical for remote training envs).

In [20]:
import pickle

# Pickle the source
data = pickle.dumps(source)
print(f"Pickled source: {len(data)} bytes")

# Unpickle
source_restored = pickle.loads(data)
print(f"Restored type: {type(source_restored).__name__}")

# Verify state was preserved
assert source_restored._collection_name == source._collection_name
assert source_restored._host == source._host
assert source_restored._port == source._port
assert source_restored._enable_bm25 == source._enable_bm25
print(f"  collection_name: {source_restored._collection_name}")
print(f"  host: {source_restored._host}")
print(f"  enable_bm25: {source_restored._enable_bm25}")

# Verify lazy state was stripped
assert source_restored._client is None, "Client should be None after unpickle"
assert source_restored._collection is None, "Collection should be None after unpickle"
assert source_restored._files is not None, "FileAwareness should be rebuilt"

print("\nPickle roundtrip: PASSED")
print("(Note: the unpickled source would need a server to reconnect — ")
print(" in-memory data is lost, which is expected and why training envs require client-server mode)")

Pickled source: 598 bytes
Restored type: ChromaChunkSource
  collection_name: integration-test
  host: None
  enable_bm25: True

Pickle roundtrip: PASSED
(Note: the unpickled source would need a server to reconnect — 
 in-memory data is lost, which is expected and why training envs require client-server mode)


## 5. SearchEnv — RL Tool Execution

Test the search environment that wraps ChromaChunkSource for RL training.
Since we don't have a Chroma server running, we test with a mocked source.

In [21]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from cgft.corpus.chroma.search import ChromaSearch
from cgft.envs.search_env import SearchEnv

search = ChromaSearch(
    collection_name=COLLECTION_NAME,
    host=CHROMA_HOST,
    port=CHROMA_PORT,
)

env = SearchEnv(search=search)
print(f"SearchEnv created, default mode: {env._default_mode}")
print(f"Tools: {list(env._tools.keys())}")


Default mode: vector
Available tools: ['search']

Tool: search
  Description: Search the corpus using full-text, vector, or hybrid search.
  Schema: {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query string.'}, 'limit': {'type': 'integer', 'description': 'Max number of results to return (default 10).'}}, 'required': ['query']}


In [22]:
# Execute the search tool as the RL agent would
search_result = asyncio.get_event_loop().run_until_complete(
    env.run_tool("rollout-1", "search", query="how to create custom environments")
)

print("Search tool output:")
print(search_result[:500])
assert "Error" not in search_result or "No results" in search_result
print("\nEnv search tool: PASSED")

Search tool output:
1.
   Content: # RL Environments  
## Overview  
Environments define how the RL agent interacts with your corpus during
training. Each environment provides tools the agent can use and a reward
function that scores the agent's responses.
2.
   Content: ### Reward Function  
Uses an LLM judge to evaluate answer correctness against ground truth.
The judge scores responses on a 0-1 scale based on factual consistency.

## Custom Environments  
You can create custom environments by extending `SearchEn

Env search tool: PASSED


In [23]:
# Test dataset_preprocess (inherited from SearchEnv)
example = {"question": "What distance metrics does Chroma support?", "answer": "cosine, l2, ip"}
preprocessed = SearchEnv.dataset_preprocess(example)
print(f"Preprocessed example:")
print(f"  prompt: {preprocessed['prompt']}")
print(f"  ground_truth: {preprocessed['ground_truth']}")
assert preprocessed["prompt"] == example["question"]
assert preprocessed["ground_truth"] == example["answer"]
print("\ndataset_preprocess: PASSED")

Preprocessed example:
  prompt: What distance metrics does Chroma support?
  ground_truth: cosine, l2, ip

dataset_preprocess: PASSED


In [24]:
# Test error handling: empty query
error_result = asyncio.get_event_loop().run_until_complete(
    env._search_tool(query="")
)
assert "Error" in error_result
print(f"Empty query error: {error_result}")
print("Error handling: PASSED")

Empty query error: Error: Missing required parameter: 'query'
Error handling: PASSED


## 6. populate_from_chunks() — Direct ChunkCollection Upload

Test the alternative populate path where you bring pre-built chunks.

In [25]:
from cgft.chunkers.models import Chunk, ChunkCollection

# Build chunks manually
manual_chunks = [
    Chunk(content="Chroma is an open-source vector database for AI applications.",
          metadata=(("file", "manual/intro.md"), ("index", 0), ("h1", "Introduction"))),
    Chunk(content="You can store embeddings and metadata together in Chroma collections.",
          metadata=(("file", "manual/intro.md"), ("index", 1), ("h1", "Introduction"))),
    Chunk(content="Chroma supports cosine similarity, L2 distance, and inner product.",
          metadata=(("file", "manual/distances.md"), ("index", 0), ("h1", "Distance Metrics"))),
]
collection = ChunkCollection(manual_chunks)

# Create a fresh source and populate from chunks
source2 = ChromaChunkSource(collection_name="manual-test")
source2.populate_from_chunks(collection, show_summary=True)

# Verify
count = source2._get_total_count()
print(f"\nTotal chunks after populate_from_chunks: {count}")
assert count == 3

# Search the manual collection
results = source2.search_text("distance metrics", top_k=2)
print(f"\nSearch 'distance metrics': {len(results)} results")
for chunk in results:
    print(f"  {chunk.content[:80]}...")

print("\npopulate_from_chunks: PASSED")


Uploading 3 chunks to Chroma collection 'manual-test'...


Uploading batches:   0%|          | 0/1 [00:00<?, ?it/s]


Upload complete! 3 chunks written to collection 'manual-test'.

Total chunks after populate_from_chunks: 3

Search 'distance metrics': 2 results
  Chroma supports cosine similarity, L2 distance, and inner product....
  Chroma is an open-source vector database for AI applications....

populate_from_chunks: PASSED


## 7. CgftPipeline Integration (Optional)

Full synthetic QA generation using Chroma as the corpus backend.
**Requires LLM API credentials** — skip if not configured.

Set the credentials below or leave blank to skip.

In [26]:
# Set these to run the CgftPipeline integration test
# Leave blank to skip
LLM_API_KEY = ""       # e.g. "sk-..." or Azure key
LLM_BASE_URL = ""      # e.g. "https://api.openai.com/v1" or Azure endpoint
API_KEY = ""            # CGFT platform API key
BASE_URL = "https://app.cgft.io"

In [27]:
if not LLM_API_KEY or not LLM_BASE_URL:
    print("SKIPPING CgftPipeline test — no LLM credentials configured.")
    print("Set LLM_API_KEY and LLM_BASE_URL above to enable.")
else:
    from cgft.qa_generation.cgft_models import (
        CgftPipelineConfig,
        CorpusConfig,
        CorpusContextConfig,
        OutputConfig,
        PlatformConfig,
        TargetsConfig,
    )
    from cgft.qa_generation.cgft_pipeline import CgftPipeline

    cfg = CgftPipelineConfig(
        platform=PlatformConfig(
            api_key=API_KEY,
            base_url=BASE_URL,
            llm_api_key=LLM_API_KEY,
            llm_base_url=LLM_BASE_URL,
        ),
        corpus=CorpusConfig(corpus_name="chroma-test", min_chunk_chars=100),
        corpus_context=CorpusContextConfig(
            description="Test documentation for ChromaDB integration",
            example_queries=["how to install", "search features", "distance metrics"],
            num_top_level_samples=2,
            num_random_samples=2,
            min_chunk_chars=100,
        ),
        targets=TargetsConfig(
            total_samples=5,
            qa_type_distribution={"lookup": 1},
        ),
        output=OutputConfig(
            dir="outputs/chroma_test",
            train_jsonl="train.jsonl",
            eval_jsonl="eval.jsonl",
        ),
    )

    cfg.resolve_api_keys()
    print(f"Generator model: {cfg.generation.llm_direct.model}")

    # Run pipeline with our Chroma source
    pipeline = CgftPipeline(cfg, source_factory=lambda _cfg: source)
    result = pipeline.run()

    train_data = result["train_dataset"]
    eval_data = result["eval_dataset"]
    stats = result["stats"]

    print(f"\nTrain samples: {len(train_data)}")
    print(f"Eval samples: {len(eval_data)}")
    print(f"Stats: {stats}")

    if train_data:
        print(f"\nSample QA pair:")
        print(f"  Q: {train_data[0].get('question', '')[:100]}")
        print(f"  A: {train_data[0].get('answer', '')[:100]}")

    print("\nCgftPipeline with Chroma: PASSED")

SKIPPING CgftPipeline test — no LLM credentials configured.
Set LLM_API_KEY and LLM_BASE_URL above to enable.


## 8. Summary

In [ ]:
print("=" * 60)
print("CHROMA INTEGRATION TEST — RESULTS")
print("=" * 60)

tests = [
    ("ChromaChunkSource init", True),
    ("populate_from_folder", source._get_total_count() > 0),
    ("sample_chunks (no filter)", len(sampled) == 3),
    ("sample_chunks (min_chars)", all(len(c.content) >= 300 for c in sampled_long)),
    ("File awareness", source._files.check()),
    ("get_top_level_chunks", len(top_level) > 0),
    ("get_chunk_with_context", "chunk_content" in context),
    ("Vector search", len(results) > 0),
    ("search_content", all(isinstance(r, str) for r in content_results)),
    ("search_text", len(text_results) > 0),
    ("search_related", isinstance(related, list)),
    ("Filtered search", all(c.get_metadata("h1") == "Search Features" for c in filtered_results)),
    ("Filter mapper", True),  # asserted inline
    ("Pickle roundtrip", source_restored._collection_name == source._collection_name),
    ("SearchEnv init", env._default_mode is not None),
    ("Env search tool", "Error" not in search_result),
    ("populate_from_chunks", source2._get_total_count() == 3),
]

passed = sum(1 for _, ok in tests if ok)
failed = len(tests) - passed

for name, ok in tests:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {name}")

print(f"\n{passed}/{len(tests)} tests passed")
if failed:
    print(f"  {failed} FAILED")
else:
    print("  All tests passed!")

# Clean up temp dir
import shutil
shutil.rmtree(DOCS_DIR, ignore_errors=True)
print(f"\nCleaned up {DOCS_DIR}")

CHROMA INTEGRATION TEST — RESULTS
  [PASS] ChromaChunkSource init
  [PASS] populate_from_folder
  [PASS] sample_chunks (no filter)
  [PASS] sample_chunks (min_chars)
  [PASS] File awareness
  [PASS] get_top_level_chunks
  [PASS] get_chunk_with_context
  [PASS] Vector search
  [PASS] search_content
  [PASS] search_text
  [PASS] search_related
  [PASS] Filtered search
  [PASS] Filter mapper
  [PASS] Pickle roundtrip
  [PASS] SearchEnv init
  [PASS] Env search tool
  [PASS] populate_from_chunks

17/17 tests passed
  All tests passed!

Cleaned up /var/folders/8w/8rh15bd90x7bn744lzw9xrhm0000gn/T/chroma_test_docs_qsxldsku


: 